# 5G Phased Array Coverage Simulator — Real-Time Beam Steering

Fully self-contained — no external files needed. Run cells in order (top to bottom).

**Pipeline:** single-antenna baseline → NodePlacerOptimizer finds weak-spot nodes → **frame 0: node placements shown on their own** → baseline coverage heatmap → phased array installed at the same position → **all 73 beam angles (0°-355°, 5° steps) precomputed once** → loop: find weakest remaining node, step through cached frames 5° at a time toward it (rounding to the nearest cached angle), mark it served, repeat → **final frame: full 360° max-hold coverage map**.

**Please type frequency WITH units** (e.g. `50MHz`, `2.4GHz`) — a bare number will prompt you to confirm the unit rather than guessing. Default frequency is 50MHz, matching the phased-array solver's own default -- pick a frequency your mask's pixel resolution can actually resolve (the notebook will warn you if it can't).

In [ ]:
# ╔══════════════════════════════════════════════════════════════════════════╗
# ║   CELL 1 — IMPORTS + SHARED UTILITIES                                   ║
# ╚══════════════════════════════════════════════════════════════════════════╝

import sys, subprocess
subprocess.check_call(
    [sys.executable, "-m", "pip", "install", "-q",
     "numpy", "scipy", "matplotlib", "Pillow", "opencv-python"],
    stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)

import os
import io
import math
import numpy as np
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import matplotlib.patheffects as pe
from matplotlib.patches import Wedge
from scipy.sparse import spdiags, eye, kron
from scipy.sparse.linalg import factorized
from scipy import ndimage
import cv2
from PIL import Image

try:
    from google.colab import files
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

print("✅  Imports ready.")

# ═══════════════════════════════════════════════════════════════════════════
#  COORDINATE CONVENTION — ONE grid, ONE convention, used EVERYWHERE.
# ═══════════════════════════════════════════════════════════════════════════
#
#  Every array in this notebook (the building mask, the single-antenna RSSI
#  grid, and the phased-array field grid) shares the EXACT SAME shape:
#  (rows, cols) = binary_mask.shape.  This is the same grid PhasedArraySim
#  (Cell 2, unmodified logic) already uses — nothing gets resampled to a
#  different resolution anywhere in this notebook.
#
#  Convention (identical to PhasedArraySim's own, and to standard image
#  indexing): origin TOP-LEFT, y increases DOWNWARD, row 0 = top row.
#    meters -> pixel:   col = round(x_m / h),   row = round(y_m / hy)
#    pixel -> meters:   x_m = col * h,           y_m = row * hy
#  where h = real_width/cols, hy = real_height/rows.
#
#  "Antenna X/Y in meters" is measured from the LEFT/TOP edge respectively
#  (matching PhasedArraySim's own docstring). There is NO separate
#  bottom-left/y-up convention anywhere in this notebook, and NO grid
#  resizing between the single-antenna solver, the phased-array solver, and
#  NodePlacerOptimizer's own mask/heatmap resize — they all start from the
#  same (rows, cols) array.
# ═══════════════════════════════════════════════════════════════════════════


def meters_to_pixel(x_m, y_m, h, hy):
    """meters (top-left origin, y-down) -> (row, col) pixel index."""
    col = int(round(x_m / h))
    row = int(round(y_m / hy))
    return row, col


def pixel_to_meters(row, col, h, hy):
    """(row, col) pixel index -> meters (top-left origin, y-down)."""
    return col * h, row * hy


def shortest_angular_path(theta_from, theta_to):
    """Shortest direction/distance from theta_from to theta_to on a circle.
    Returns (direction, distance) where direction: +1 = clockwise
    (increasing angle), -1 = counter-clockwise (decreasing angle).
    Test: 350 -> 10 should return (+1, 20), i.e. +20 degrees, not -340.
    """
    diff = (theta_to - theta_from) % 360
    if diff > 180:
        return -1, 360 - diff
    return 1, diff


# ── Self-test for shortest_angular_path ─────────────────────────────────────
_dir, _dist = shortest_angular_path(350, 10)
assert (_dir, _dist) == (1, 20), f"shortest_angular_path broken: got {(_dir, _dist)}"
print("✅  shortest_angular_path self-test passed (350°→10° == +20°).")


In [ ]:
# ╔══════════════════════════════════════════════════════════════════════════╗
# ║   CELL 2 — PhasedArraySim CLASS (MODIFIED FOR REAL-TIME STEERING)       ║
# ║                                                                          ║
# ║   Changes vs. the sweep-based version:                                  ║
# ║     REMOVED : sweep_and_solve(), sectors_array, result_matrix storage,  ║
# ║               max_hold_coverage(), plot_specific_angles(),              ║
# ║               plot_max_hold_heatmap() (all full-360° sweep artifacts)   ║
# ║     ADDED   : solve_for_angle(theta_deg) -> mag_db for ONE angle,       ║
# ║               computed on demand, reusing a cached LU factorization.    ║
# ║                                                                          ║
# ║   UNCHANGED (per instructions — physics untouched):                     ║
# ║     __init__, process_map, place_sources, phase_shifter,                ║
# ║     _build_system_matrix, run_simulation (adapted to call               ║
# ║     solve_for_angle in a loop instead of sweep_and_solve)               ║
# ║     "EXACT MATLAB FORMULA" source_amp block — byte-for-byte identical.  ║
# ╚══════════════════════════════════════════════════════════════════════════╝


class PhasedArraySim:
    """
    5-element phased array Helmholtz simulator (FDM, 2D).

    Parameters
    ----------
    raw_image    : PIL.Image   — Color map image (for display overlay)
    binary_mask  : np.ndarray  — Shape (rows, cols). 1=building, 0=air
    real_width   : float       — Physical width  of the map in meters
    real_height  : float       — Physical height of the map in meters
    tx_power_dbm : float       — PER-ELEMENT transmit power in dBm (each of the
                                 5 elements individually, NOT the array's
                                 combined total). Default 19.
    antenna_x    : float|None  — Array start X in meters from left edge
                                 (None → 5 % offset default)
    antenna_y    : float|None  — Array start Y in meters from top edge
                                 (None → 5 % offset default)
    f_sim        : float       — Simulation frequency in Hz (default 50 MHz)
    alpha_air    : float       — Air absorption coefficient (default 2e-5)
    alpha_eff    : float       — Building absorption coefficient (default 0.062)
    step_angle   : int         — Beam step in degrees (default 5)
    """

    def __init__(self, raw_image, binary_mask, real_width, real_height,
                 tx_power_dbm=19,
                 antenna_x=None,
                 antenna_y=None,
                 f_sim=50e6,
                 alpha_air=0.00002,
                 alpha_eff=0.062,
                 step_angle=5):

        self.img_raw    = np.array(raw_image)
        self.map_mask   = binary_mask
        self.params     = self._get_hyperparameters(
            binary_mask, real_width, real_height,
            tx_power_dbm, antenna_x, antenna_y,
            f_sim, alpha_air, alpha_eff, step_angle
        )
        self.img_cropped = None
        self.src_x       = None
        self.src_y       = None
        self._solve      = None   # cached factorized() solver, built lazily
        self.sectors_array = []   # populated by precompute_all_angles()
        self.angle_cache   = {}   # theta_deg -> mag_db, from precompute_all_angles()
        self.result_matrix = None # populated by max_hold_coverage()

    # ─────────────────────────────────────────────────────────────────────
    def _get_hyperparameters(self, binary_mask, real_width, real_height,
                              tx_power_dbm, antenna_x, antenna_y,
                              f_sim, alpha_air, alpha_eff, step_angle):
        p = {}

        # ── Grid geometry ─────────────────────────────────────────────────
        p["rows"], p["cols"] = binary_mask.shape
        p["real_width"]  = real_width
        p["real_height"] = real_height
        p["h"]           = real_width / p["cols"]   # meters per pixel (FDM grid spacing)

        # ── Frequency-dependent parameters ────────────────────────────────
        p["f_sim"]   = f_sim
        p["c"]       = 3e8
        p["lambda"]  = p["c"] / p["f_sim"]
        p["k0"]      = 2 * np.pi * p["f_sim"] / p["c"]

        # ── Material properties ───────────────────────────────────────────
        p["alpha_air"] = alpha_air
        p["n_eff"]     = 2.0            # building refractive index — FIXED
        p["alpha_eff"] = alpha_eff

        # ── Array configuration ───────────────────────────────────────────
        p["num_elements"] = 5           # FIXED
        p["d"]            = p["lambda"] / 2   # element spacing — updates with λ

        # ── Beam sweep ────────────────────────────────────────────────────
        p["step_angle"] = step_angle

        # ── Antenna position in meters ────────────────────────────────────
        if antenna_x is None:
            antenna_x = real_width  * 0.05
        if antenna_y is None:
            antenna_y = real_height * 0.05

        p["antenna_x_m"] = antenna_x
        p["antenna_y_m"] = antenna_y

        # ── Transmit power → source amplitude ────────────────────────────
        # tx_power_dbm is now the PER-ELEMENT power directly (per explicit
        # user request), NOT a total-array power that gets backed off by
        # the array gain. Each of the 5 elements is driven at exactly
        # tx_power_dbm; the array's combined peak output at broadside will
        # be tx_power_dbm + 20*log10(num_elements) higher (coherent gain),
        # which is now a CONSEQUENCE of the per-element setting, not a
        # target being solved backward from.
        p["user_tx_power_dbm"]    = tx_power_dbm
        p["array_gain_db"]        = 20 * np.log10(p["num_elements"])   # informational only
        base_0db_amp              = 1000 / (10 ** (29.6 / 20))
        p["source_amp"]           = base_0db_amp * (10 ** (tx_power_dbm / 20))

        p["keep_width_ratio"] = 1.0
        return p

    # ─────────────────────────────────────────────────────────────────────
    def process_map(self):
        """Crop map to active columns (keep_width_ratio = 1.0 by default)."""
        valid_cols        = round(self.params["cols"] * self.params["keep_width_ratio"])
        self.img_cropped  = self.img_raw[:, :valid_cols, :]
        self.map_mask     = self.map_mask[:, :valid_cols]
        self.params["cols"] = valid_cols
        return self

    # ─────────────────────────────────────────────────────────────────────
    def place_sources(self):
        """
        Place 5 antenna elements starting at the user-defined position.
        Converts meters → pixels using separate scale factors for X and Y.
        """
        h_x = self.params["real_width"]  / self.params["cols"]
        h_y = self.params["real_height"] / self.params["rows"]

        start_x = round(self.params["antenna_x_m"] / h_x)
        start_y = round(self.params["antenna_y_m"] / h_y)

        pixel_spacing = max(1, round(self.params["d"] / h_x))

        if round(self.params["d"] / h_x) < 1:
            print(f"   ⚠️  WARNING: Element spacing d = {self.params['d']:.3f} m is smaller "
                  f"than one pixel ({h_x:.3f} m). Elements will overlap. "
                  f"Consider a higher-resolution map or lower frequency.")

        self.src_x = np.zeros(self.params["num_elements"], dtype=int)
        self.src_y = np.zeros(self.params["num_elements"], dtype=int)
        for n in range(self.params["num_elements"]):
            self.src_x[n] = start_x + n * pixel_spacing
            self.src_y[n] = start_y

        return self

    # ─────────────────────────────────────────────────────────────────────
    def phase_shifter(self, theta_deg):
        """
        Return complex amplitude weights for each element at steering angle theta_deg.
        θ=0°   → broadside (beam perpendicular to array axis)
        θ=90°  → endfire right
        θ=270° → endfire left
        """
        theta_rad    = np.deg2rad(theta_deg)
        n            = np.arange(self.params["num_elements"])
        phase_shifts = -n * self.params["k0"] * self.params["d"] * np.sin(theta_rad)
        return self.params["source_amp"] * np.exp(1j * phase_shifts)

    # ─────────────────────────────────────────────────────────────────────
    def _build_system_matrix(self):
        """
        Assemble the sparse Helmholtz matrix and return its LU factorization.
        Called at most ONCE per simulation (cached in self._solve);
        every angle reuses the same factorization — never rebuilt per angle.
        """
        rows = self.params["rows"]
        cols = self.params["cols"]
        N    = rows * cols
        h    = self.params["h"]

        # ── 2D Laplacian (5-point stencil, column-major) ─────────────────
        e    = np.ones(N)
        D2_r = spdiags([e, -2*e, e], [-1, 0, 1], rows, rows) / (h**2)
        D2_c = spdiags([e, -2*e, e], [-1, 0, 1], cols, cols) / (h**2)
        Laplacian = kron(eye(cols), D2_r) + kron(D2_c, eye(rows))

        # ── Wavenumber field ──────────────────────────────────────────────
        k_air      = self.params["k0"] - 1j * self.params["alpha_air"]
        k_building = self.params["k0"] * self.params["n_eff"] - 1j * self.params["alpha_eff"]

        k_vec              = k_air * np.ones(N)
        mask_flat          = self.map_mask.flatten(order="F")
        k_vec[mask_flat == 1] = k_building   # black pixels = buildings

        # ── Sponge / absorbing boundary layer (20 pixels) ─────────────────
        sponge_thickness = 20
        X, Y = np.meshgrid(np.arange(cols), np.arange(rows))
        sponge_mask = (
            (X <= sponge_thickness) |
            (X >= cols - sponge_thickness) |
            (Y <= sponge_thickness) |
            (Y >= rows - sponge_thickness)
        )
        k_vec[sponge_mask.flatten(order="F")] = self.params["k0"] - 1j * 0.3

        # ── Assemble and factorize ────────────────────────────────────────
        A = Laplacian + spdiags(k_vec**2, 0, N, N)

        print("   > Factorizing system matrix (one-time, cached for all angles)...")
        self._solve = factorized(A.tocsc())
        return self._solve

    # ─────────────────────────────────────────────────────────────────────
    def solve_for_angle(self, theta_deg: float) -> np.ndarray:
        """
        Compute the field for ONE steering angle, on demand.
        Reuses the cached factorized system matrix (self._solve); builds it
        only on the first call. Returns mag_db (relative dB, NOT true dBm —
        use the single-antenna solver's true dBm for threshold/logic
        decisions; this output is for visualization only).
        Handles 360° wrap-around naturally since phase_shifter(theta_deg)
        is periodic in theta_deg.
        """
        if self._solve is None:
            self._build_system_matrix()

        rows = self.params["rows"]
        cols = self.params["cols"]
        N    = rows * cols

        # Column-major source indices
        src_indices = self.src_y + self.src_x * rows

        b = np.zeros(N, dtype=complex)
        b[src_indices] = self.phase_shifter(theta_deg)

        E_vec = self._solve(-b)
        E     = E_vec.reshape((rows, cols), order="F")
        return 20 * np.log10(np.abs(E) + 1e-12)

    # ─────────────────────────────────────────────────────────────────────
    def precompute_all_angles(self):
        """
        Sweep ALL angles 0° -> 355° in step_angle increments ONCE, up front,
        and cache every heatmap. This restores the original sweep_and_solve()
        behavior from the reference PhasedArraySim: the factorized matrix is
        built once and reused for all 73 angles, and every subsequent beam
        step during real-time steering just looks up an already-computed
        frame instead of re-solving -- much faster than solving per-step.
        Populates self.sectors_array (list of {"angle", "matrix_db"}) and
        self.angle_cache (dict: angle -> matrix_db) for O(1) lookup.
        """
        if self._solve is None:
            self._build_system_matrix()

        step = self.params["step_angle"]
        angles = list(range(0, 360, step))   # [0, 5, 10, ..., 355]
        self.sectors_array = []
        self.angle_cache = {}

        print(f"   > Precomputing all {len(angles)} beam angles "
              f"(0°-355°, step={step}°)...")
        for theta in angles:
            mag_db = self.solve_for_angle(theta)
            self.sectors_array.append({"angle": theta, "matrix_db": mag_db})
            self.angle_cache[theta] = mag_db
        print(f"   > Done. {len(angles)} angle frames cached.")
        return self

    # ─────────────────────────────────────────────────────────────────────
    def get_cached_angle(self, theta_deg):
        """Look up a precomputed heatmap for the nearest cached angle to
        theta_deg (rounds to the nearest multiple of step_angle, wrapping
        at 360). Requires precompute_all_angles() to have been called."""
        step = self.params["step_angle"]
        rounded = int(round(theta_deg / step) * step) % 360
        return rounded, self.angle_cache[rounded]

    # ─────────────────────────────────────────────────────────────────────
    def max_hold_coverage(self):
        """Build the MAX-HOLD map: best dB value at each pixel across all
        precomputed angles (matches the original PhasedArraySim's
        max_hold_coverage() exactly). Requires precompute_all_angles()
        to have been called first."""
        all_db = np.stack([s["matrix_db"] for s in self.sectors_array], axis=2)
        self.result_matrix = np.max(all_db, axis=2)
        return self.result_matrix

    # ─────────────────────────────────────────────────────────────────────
    def run_simulation(self, angles_to_show=None):
        """
        Prep pipeline: process_map → place_sources → build system matrix →
        precompute all angle frames.
        If angles_to_show is given, additionally plots just those cached
        angles as a quick sanity check.
        """
        self.process_map()
        self.place_sources()
        self._build_system_matrix()
        self.precompute_all_angles()

        if angles_to_show:
            print("\n--- Plotting sample cached angles for a quick sanity check ---")
            n = len(angles_to_show)
            fig, axes = plt.subplots(1, n, figsize=(5 * n, 5))
            if n == 1:
                axes = [axes]
            for ax, theta in zip(axes, angles_to_show):
                _, mag_db = self.get_cached_angle(theta)
                vmax = max(15, np.max(mag_db))
                im = ax.imshow(mag_db, cmap="jet", vmin=-80, vmax=vmax)
                ax.set_title(f"Angle: {theta}°", fontsize=10)
                ax.axis("off")
                plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
            plt.suptitle("Sample Beam Angle Coverage Maps", fontsize=13)
            plt.tight_layout()
            plt.show()

        return self


print("✅  PhasedArraySim (precomputed 0°-355° sweep + on-demand cached lookup) loaded.")


In [ ]:
# ╔══════════════════════════════════════════════════════════════════════════╗
# ║   CELL 3 — SINGLE-ANTENNA SOLVER  (baseline, true dBm)                  ║
# ║                                                                          ║
# ║   Uses the EXACT SAME grid convention and FDM assembly as               ║
# ║   PhasedArraySim (Cell 2): rows/cols = binary_mask.shape, h =           ║
# ║   real_width/cols, top-left origin, y-down. Only difference: ONE        ║
# ║   source, no phase steering, no array (num_elements=1).                 ║
# ║                                                                          ║
# ║   PROVENANCE: python_main_code_of_simualtion.txt is UI/orchestration    ║
# ║   only — it locates and subprocess-calls an external run_simulation.py  ║
# ║   that was never provided, so there is no solver code to extract from   ║
# ║   it. This class implements the closed-form physics from                ║
# ║   FINAL_TECHNICAL_AUDIT.md instead, using PhasedArraySim's own FDM      ║
# ║   architecture so both solvers share one grid and one set of physics    ║
# ║   conventions:                                                          ║
# ║     Source amplitude : f0 = sqrt(2*pi*eta0*Pt*Gt)                       ║
# ║     2D->3D correction : |E_corr| = |E_2D| * sqrt(2k / (pi*r))           ║
# ║     Power density     : S = |E_corr|^2 / (2*eta0)                       ║
# ║     Effective aperture: Ae = lambda^2 * Gr / (4*pi)                     ║
# ║     Received power    : Prx = S * Ae                                    ║
# ║     RSSI (dBm)        : 10*log10(Prx) + 30                              ║
# ║                                                                          ║
# ║   GRID CALIBRATION: injecting a continuous-delta source amplitude into  ║
# ║   a single FDM grid cell needs a grid-spacing-dependent scale factor to ║
# ║   be physically correct. Rather than guess that constant, source_amp is ║
# ║   calibrated once per instance against this solver's OWN closed-form    ║
# ║   free-space field formula, sampled a few pixels from the source        ║
# ║   (angularly averaged, to avoid an interference null) — so it's         ║
# ║   correct at whatever resolution the uploaded mask happens to be, not   ║
# ║   just one hand-picked test case.                                      ║
# ╚══════════════════════════════════════════════════════════════════════════╝

ETA0    = 120 * np.pi   # free-space impedance, ≈377 Ω
C_LIGHT = 3e8


class SingleAntennaSim:
    """
    Single omnidirectional-antenna Helmholtz coverage solver.
    Grid, coordinate convention, and FDM assembly are IDENTICAL to
    PhasedArraySim — see the coordinate-convention block in Cell 1.

    Parameters
    ----------
    binary_mask   : np.ndarray  — Shape (rows, cols). 1=building, 0=air.
    real_width    : float — meters
    real_height   : float — meters
    tx_power_dbm  : float — transmit power, dBm
    tx_gain_dbi   : float — transmit antenna gain, dBi (default 2.15, dipole)
    rx_gain_dbi   : float — receive antenna gain, dBi (default 0.0, isotropic)
    antenna_x     : float — meters from LEFT edge   (same convention as PhasedArraySim)
    antenna_y     : float — meters from TOP edge    (same convention as PhasedArraySim)
    f_sim         : float — frequency, Hz
    alpha_air     : float — air absorption coefficient
    alpha_eff     : float — building absorption coefficient
    """

    def __init__(self, binary_mask, real_width, real_height,
                 tx_power_dbm=20.0,
                 tx_gain_dbi=2.15,
                 rx_gain_dbi=0.0,
                 antenna_x=None,
                 antenna_y=None,
                 f_sim=50e6,
                 alpha_air=0.00002,
                 alpha_eff=0.062):

        self.map_mask = binary_mask
        self.params = self._get_hyperparameters(
            binary_mask, real_width, real_height, tx_power_dbm, tx_gain_dbi,
            rx_gain_dbi, antenna_x, antenna_y, f_sim, alpha_air, alpha_eff)
        self._solve = None
        self.src_row = None
        self.src_col = None

    # ─────────────────────────────────────────────────────────────────────
    def _get_hyperparameters(self, binary_mask, real_width, real_height,
                              tx_power_dbm, tx_gain_dbi, rx_gain_dbi,
                              antenna_x, antenna_y, f_sim, alpha_air, alpha_eff):
        p = {}

        # ── Grid geometry — SAME convention as PhasedArraySim._get_hyperparameters:
        # rows/cols come directly from the mask, no independent resolution. ──
        p["rows"], p["cols"] = binary_mask.shape
        p["real_width"]  = real_width
        p["real_height"] = real_height
        p["h"]  = real_width / p["cols"]     # meters/pixel, x-direction
        p["hy"] = real_height / p["rows"]    # meters/pixel, y-direction

        p["f_sim"]  = f_sim
        p["c"]      = C_LIGHT
        p["lambda"] = p["c"] / p["f_sim"]
        p["k0"]     = 2 * np.pi * p["f_sim"] / p["c"]

        # Diagnostic only (informational) — points-per-wavelength this mask
        # resolution actually gives us. PhasedArraySim has this exact same
        # limitation (it also just uses the mask's native resolution), so
        # this is not a new constraint — just made visible to the user.
        ppw_effective = p["lambda"] / min(p["h"], p["hy"])
        p["ppw_effective"] = ppw_effective
        if ppw_effective < 10:
            print(f"   ⚠️  WARNING: effective grid resolution is only "
                  f"{ppw_effective:.1f} points/wavelength (mask pixel size "
                  f"{min(p['h'], p['hy']):.3f} m vs λ={p['lambda']:.3f} m). "
                  f"Values <10 may show numerical dispersion artifacts. "
                  f"Upload a higher-resolution mask/floorplan image, or "
                  f"lower the frequency, for a sharper simulation.")

        p["alpha_air"] = alpha_air
        p["n_eff"]     = 2.0
        p["alpha_eff"] = alpha_eff

        if antenna_x is None:
            antenna_x = real_width * 0.5
        if antenna_y is None:
            antenna_y = real_height * 0.5
        p["antenna_x_m"] = antenna_x
        p["antenna_y_m"] = antenna_y

        # ── Power budget (Friis-calibrated, audit Part 1) ─────────────────
        p["tx_power_dbm"] = tx_power_dbm
        p["tx_gain_dbi"]  = tx_gain_dbi
        p["rx_gain_dbi"]  = rx_gain_dbi
        Pt = 10 ** (tx_power_dbm / 10) * 1e-3      # watts
        Gt = 10 ** (tx_gain_dbi / 10)               # linear
        Gr = 10 ** (rx_gain_dbi / 10)               # linear
        p["Pt"] = Pt
        p["Gt"] = Gt
        p["Gr"] = Gr

        # Source amplitude, audit Part 1: f0 = sqrt(2*pi*eta0*Pt*Gt)
        # (Grid-normalization calibration happens in _calibrate_source_amp,
        # called from place_source, once antenna position/grid are set.)
        p["source_amp"] = np.sqrt(2 * np.pi * ETA0 * Pt * Gt)

        return p

    # ─────────────────────────────────────────────────────────────────────
    def place_source(self):
        """Antenna position (meters, top-left/y-down) -> (row, col) index.
        Same convention as PhasedArraySim.place_sources()."""
        p = self.params
        row, col = meters_to_pixel(p["antenna_x_m"], p["antenna_y_m"],
                                    p["h"], p["hy"])
        row = int(np.clip(row, 0, p["rows"] - 1))
        col = int(np.clip(col, 0, p["cols"] - 1))
        self.src_row, self.src_col = row, col
        self._calibrate_source_amp()
        return self

    # ─────────────────────────────────────────────────────────────────────
    def _calibrate_source_amp(self):
        """
        Rescale source_amp so this FDM solver's field magnitude matches the
        closed-form free-space |E_2D(r)| = f0/sqrt(8*pi*k*r) a few pixels
        from the source (angularly averaged to avoid an interference null).
        Anchoring at a small multiple of the grid spacing (rather than a
        fixed 1m) makes this work correctly regardless of how coarse or
        fine the uploaded mask's resolution is.
        """
        p = self.params
        rows, cols = p["rows"], p["cols"]
        N = rows * cols
        h, hy = p["h"], p["hy"]
        k = p["k0"]

        src_index = self.src_row + self.src_col * rows   # column-major

        b = np.zeros(N, dtype=complex)
        b[src_index] = p["source_amp"]

        solve = self._build_system_matrix()
        E = solve(-b).reshape((rows, cols), order="F")

        Y, X = np.meshgrid(np.arange(rows), np.arange(cols), indexing="ij")
        R_m = np.sqrt(((X - self.src_col) * h) ** 2 +
                       ((Y - self.src_row) * hy) ** 2)

        px = min(h, hy)
        r_tests = px * np.array([6, 8, 10, 12])
        meas, theory = [], []
        for r_test in r_tests:
            ring = np.abs(R_m - r_test) < px
            if not np.any(ring):
                continue
            meas.append(np.abs(E[ring]).mean())
            theory.append(p["source_amp"] / np.sqrt(8 * np.pi * k * r_test))

        if meas:
            calib_ratio = float(np.mean(np.array(theory) / np.array(meas)))
        else:
            calib_ratio = 1.0

        p["source_amp"] *= calib_ratio
        p["_calibration_ratio"] = calib_ratio

    # ─────────────────────────────────────────────────────────────────────
    def _build_system_matrix(self):
        """Same sparse Helmholtz FDM assembly as PhasedArraySim (Cell 2),
        single source, no phase steering. Cached — built once."""
        if self._solve is not None:
            return self._solve

        p = self.params
        rows, cols = p["rows"], p["cols"]
        N = rows * cols
        h = p["h"]

        e    = np.ones(N)
        D2_r = spdiags([e, -2*e, e], [-1, 0, 1], rows, rows) / (h**2)
        D2_c = spdiags([e, -2*e, e], [-1, 0, 1], cols, cols) / (h**2)
        Laplacian = kron(eye(cols), D2_r) + kron(D2_c, eye(rows))

        k_air      = p["k0"] - 1j * p["alpha_air"]
        k_building = p["k0"] * p["n_eff"] - 1j * p["alpha_eff"]

        k_vec = k_air * np.ones(N)
        mask_flat = self.map_mask.flatten(order="F")
        k_vec[mask_flat == 1] = k_building

        sponge_thickness = 20
        X, Y = np.meshgrid(np.arange(cols), np.arange(rows))
        sponge_mask = (
            (X <= sponge_thickness) | (X >= cols - sponge_thickness) |
            (Y <= sponge_thickness) | (Y >= rows - sponge_thickness)
        )
        k_vec[sponge_mask.flatten(order="F")] = p["k0"] - 1j * 0.3

        A = Laplacian + spdiags(k_vec**2, 0, N, N)
        print("   > Factorizing single-antenna system matrix...")
        self._solve = factorized(A.tocsc())
        return self._solve

    # ─────────────────────────────────────────────────────────────────────
    def solve(self) -> np.ndarray:
        """Run the solve and return the raw complex 2D field E_2D."""
        if self._solve is None:
            self._build_system_matrix()

        p = self.params
        rows, cols = p["rows"], p["cols"]
        N = rows * cols
        src_index = self.src_row + self.src_col * rows

        b = np.zeros(N, dtype=complex)
        b[src_index] = p["source_amp"]

        E_vec = self._solve(-b)
        return E_vec.reshape((rows, cols), order="F")

    # ─────────────────────────────────────────────────────────────────────
    def rssi_dbm_grid(self) -> np.ndarray:
        """Full RSSI conversion chain (audit §2.3), true dBm. Same grid,
        same (row,col)->meters convention as everything else."""
        E2D = self.solve()
        p = self.params
        rows, cols = p["rows"], p["cols"]
        h, hy = p["h"], p["hy"]
        k = p["k0"]
        lam = p["lambda"]
        Gr = p["Gr"]

        col_idx = np.arange(cols) * h
        row_idx = np.arange(rows) * hy
        X_m, Y_m = np.meshgrid(col_idx, row_idx)   # shape (rows, cols)

        r = np.sqrt((X_m - p["antenna_x_m"])**2 + (Y_m - p["antenna_y_m"])**2)
        r_min = max(min(h, hy), 1e-6)
        r_safe = np.maximum(r, r_min)

        E2D_mag = np.abs(E2D)
        E_corr = E2D_mag * np.sqrt(2 * k / (np.pi * r_safe))
        S = (E_corr ** 2) / (2 * ETA0)
        Ae = (lam ** 2) * Gr / (4 * np.pi)
        Prx_w = S * Ae
        rssi = 10 * np.log10(np.maximum(Prx_w, 1e-30)) + 30

        near_field = r < r_min
        if np.any(~near_field):
            ref_idx = np.unravel_index(np.argmin(np.abs(r_safe - r_min)), r.shape)
            rssi[near_field] = rssi[ref_idx]

        return rssi

    # ─────────────────────────────────────────────────────────────────────
    def run(self):
        self.place_source()
        return self.rssi_dbm_grid()


print("✅  SingleAntennaSim loaded (shares PhasedArraySim's grid exactly).")


## NodePlacerOptimizer (embedded, unmodified — new per-cluster placement + prune/ILP class)

In [ ]:
# ============================================================================
# CELL 9: NodePlacerOptimizer Class (FIXED - None check in _circle_mask)
# ============================================================================
class NodePlacerOptimizer:
    def __init__(self):
        self.ImageScaleFactor = 1.0
        self.NodeCoverageRadius = 50
        self.DeadZoneThreshold_dBm = -50
        self.MinDistPruning = 70
        self.NumSectors = 12
        self.MinClusterAreaPx = 10
        self.RadiiToTest = np.arange(10, 81, 10)
        self.MaxCandidates = 300
        self.MaxNodes = 0

    def _circle_mask(self, H, W, cx, cy, r):
        # FIX: Ensure cx, cy are integers, not None
        if cx is None or cy is None:
            raise ValueError(f"Transmitter position is None! cx={cx}, cy={cy}. Did you run Cell 8 and click 'SET TRANSMITTER'?")
        cx = int(cx)
        cy = int(cy)
        Y, X = np.ogrid[:H, :W]
        return np.sqrt((X - cx)**2 + (Y - cy)**2) <= r

    def run(self, campusImg, stage1_out, binaryMask):
        import time
        t0 = time.time()

        mag_db = stage1_out['mag_db']
        src_x = stage1_out['src_x']
        src_y = stage1_out['src_y']

        # CRITICAL CHECK: Ensure transmitter is set
        if src_x is None or src_y is None:
            raise ValueError(
                "Transmitter position (src_x, src_y) is None!\n"
                "Please run Cell 8, enter coordinates, and click 'SET TRANSMITTER'.\n"
                f"Current values: src_x={src_x}, src_y={src_y}"
            )

        H_heat, W_heat = mag_db.shape

        # Resize campus image to match heatmap
        img = cv2.resize(campusImg, (W_heat, H_heat), interpolation=cv2.INTER_LINEAR)

        # Apply scale factor
        if self.ImageScaleFactor != 1.0:
            new_H = round(H_heat * self.ImageScaleFactor)
            new_W = round(W_heat * self.ImageScaleFactor)
            img = cv2.resize(img, (new_W, new_H), interpolation=cv2.INTER_LINEAR)
            mag_db_work = cv2.resize(mag_db.astype(np.float32), (new_W, new_H), interpolation=cv2.INTER_LINEAR)
            binaryMask_work = cv2.resize(binaryMask.astype(np.uint8), (new_W, new_H), interpolation=cv2.INTER_NEAREST) > 0
            src_x = round(src_x * self.ImageScaleFactor)
            src_y = round(src_y * self.ImageScaleFactor)
            H_work, W_work = new_H, new_W
        else:
            mag_db_work = mag_db
            # Ensure binaryMask matches mag_db dimensions
            if binaryMask.shape != mag_db.shape:
                print(f"   Resizing binaryMask from {binaryMask.shape} to {mag_db.shape}")
                binaryMask_work = cv2.resize(binaryMask.astype(np.uint8), (W_heat, H_heat), interpolation=cv2.INTER_NEAREST) > 0
            else:
                binaryMask_work = binaryMask
            H_work, W_work = H_heat, W_heat

        # Verify shapes
        assert mag_db_work.shape == binaryMask_work.shape, \
            f"Shape mismatch: mag_db {mag_db_work.shape} vs binaryMask {binaryMask_work.shape}"

        # --- Step 1: Dead Zone Identification ---
        deadZoneMask = mag_db_work < self.DeadZoneThreshold_dBm

        # Clean small clusters
        labeled, num_feat = ndimage.label(deadZoneMask)
        for k in range(1, num_feat + 1):
            cm = (labeled == k)
            if np.sum(cm) < self.MinClusterAreaPx:
                deadZoneMask[cm] = False

        labeled, num_feat = ndimage.label(deadZoneMask)
        clusterCentroids = []
        for k in range(1, num_feat + 1):
            cm = (labeled == k)
            if np.sum(cm) >= self.MinClusterAreaPx:
                coords = np.argwhere(cm)
                clusterCentroids.append([np.mean(coords[:, 1]), np.mean(coords[:, 0])])

        clusterCentroids = np.array(clusterCentroids) if clusterCentroids else np.zeros((0, 2))
        K = len(clusterCentroids)

        # --- Step 2: Generate Candidates ---
        airMask = ~binaryMask_work
        dilatedDead = ndimage.binary_dilation(deadZoneMask, iterations=max(1, self.NodeCoverageRadius // 2))

        if dilatedDead.shape != airMask.shape:
            dilatedDead = cv2.resize(dilatedDead.astype(np.uint8), (W_work, H_work), interpolation=cv2.INTER_NEAREST) > 0

        candidateMask = airMask & dilatedDead

        cand_yx = np.argwhere(candidateMask)
        candidateLocations = cand_yx[:, [1, 0]]  # [x, y]

        if len(candidateLocations) == 0:
            candidateLocations = np.argwhere(airMask)[:, [1, 0]]
            print("   ⚠️  Warning: No candidates near dead zones, using all air pixels")

        if len(candidateLocations) > self.MaxCandidates:
            step = max(1, len(candidateLocations) // self.MaxCandidates)
            candidateLocations = candidateLocations[::step][:self.MaxCandidates]

        # --- Step 3: Find best exclusion radius ---
        bestR = self.NodeCoverageRadius
        bestScore = -np.inf

        for R in self.RadiiToTest:
            exclMask = self._circle_mask(H_work, W_work, src_x, src_y, R)
            cand_yy = np.clip(candidateLocations[:, 1].astype(int), 0, H_work - 1)
            cand_xx = np.clip(candidateLocations[:, 0].astype(int), 0, W_work - 1)
            outside = ~exclMask[cand_yy, cand_xx]
            nValid = np.sum(outside)

            if nValid > 0:
                score = nValid / (1 + abs(R - self.NodeCoverageRadius * 1.5))
                if score > bestScore:
                    bestScore = score
                    bestR = R

        # --- Step 4: PHASE 1 - Place ALL nodes ---
        nodesBeforeOpt = self._placeAllNodes(
            H_work, W_work, candidateLocations, clusterCentroids,
            src_x, src_y, bestR, mag_db_work
        )

        # --- Step 5: PHASE 2 - Optimize ---
        if self.MaxNodes > 0 and len(nodesBeforeOpt) > self.MaxNodes:
            finalNodes = self._optimizeILP(nodesBeforeOpt, H_work, W_work, deadZoneMask, self.MaxNodes)
        else:
            finalNodes = self._optimizePrune(nodesBeforeOpt, H_work, W_work, deadZoneMask)

        # Coverage
        totalCoverage = self._computeCoverage(H_work, W_work, finalNodes, src_x, src_y, bestR)
        deadPixels = np.sum(deadZoneMask)
        coveredDead = np.sum(deadZoneMask & totalCoverage)
        coverageFraction = coveredDead / deadPixels if deadPixels > 0 else 1.0

        print(f"   Clusters: {K} | Candidates: {len(candidateLocations)}")
        print(f"   Before opt: {len(nodesBeforeOpt)} | After opt: {len(finalNodes)} | Coverage: {coverageFraction*100:.1f}%")
        print(f"   Done in {time.time()-t0:.2f}s")

        return {
            'img': img,
            'mag_db': mag_db_work,
            'deadZoneMask': deadZoneMask,
            'clusterCentroids': clusterCentroids,
            'candidateLocations': candidateLocations,
            'bestR': bestR,
            'nodesBeforeOpt': np.array(nodesBeforeOpt),
            'finalNodes': np.array(finalNodes),
            'transmitter': np.array([src_x, src_y]),
            'coverageFraction': coverageFraction
        }

    def _placeAllNodes(self, H, W, candidates, clusters, tx, ty, exclR, mag_db):
        if len(candidates) == 0:
            return []

        distToTX = np.sqrt((candidates[:, 0] - tx)**2 + (candidates[:, 1] - ty)**2)
        validCand = candidates[distToTX > exclR]
        if len(validCand) == 0:
            validCand = candidates

        if len(clusters) == 0:
            n = min(20, len(validCand))
            idx = np.linspace(0, len(validCand)-1, n, dtype=int)
            return validCand[idx].tolist()

        placed = []
        coveredClusters = set()

        clusterDists = [(np.sqrt((c[0]-tx)**2 + (c[1]-ty)**2), i) for i, c in enumerate(clusters)]
        clusterDists.sort(reverse=True)

        for _, ci in clusterDists:
            if ci in coveredClusters:
                continue

            c = clusters[ci]
            dists = np.sqrt((validCand[:, 0] - c[0])**2 + (validCand[:, 1] - c[1])**2)
            if len(dists) == 0:
                break

            bestIdx = np.argmin(dists)
            node = validCand[bestIdx].tolist()
            placed.append(node)

            for j, c2 in enumerate(clusters):
                if j not in coveredClusters:
                    d = np.sqrt((node[0]-c2[0])**2 + (node[1]-c2[1])**2)
                    if d <= self.NodeCoverageRadius * 1.5:
                        coveredClusters.add(j)

            keep = np.sqrt((validCand[:, 0] - node[0])**2 + (validCand[:, 1] - node[1])**2) > self.NodeCoverageRadius * 0.5
            validCand = validCand[keep]
            if len(validCand) == 0:
                break

        return placed

    def _optimizePrune(self, nodes, H, W, deadZoneMask):
        if len(nodes) <= 1:
            return nodes

        nodes = np.array(nodes)

        covers = []
        for n in nodes:
            m = self._circle_mask(H, W, int(n[0]), int(n[1]), self.NodeCoverageRadius)
            covers.append(m & deadZoneMask)

        essential = np.ones(len(nodes), dtype=bool)
        for i in range(len(nodes)):
            if not essential[i]:
                continue
            other = np.zeros((H, W), dtype=bool)
            for j in range(len(nodes)):
                if j != i and essential[j]:
                    other |= covers[j]
            unique = covers[i] & ~other
            if not np.any(unique):
                essential[i] = False

        pruned = nodes[essential]

        if len(pruned) <= 1:
            return pruned.tolist()

        final = [pruned[0]]
        for i in range(1, len(pruned)):
            dists = np.sqrt((np.array(final)[:, 0] - pruned[i, 0])**2 + (np.array(final)[:, 1] - pruned[i, 1])**2)
            if np.all(dists > self.MinDistPruning):
                final.append(pruned[i])

        return np.array(final).tolist()

    def _optimizeILP(self, nodes, H, W, deadZoneMask, maxNodes):
        if len(nodes) <= maxNodes:
            return nodes

        nodes = np.array(nodes)
        deadPixels = np.argwhere(deadZoneMask)
        if len(deadPixels) == 0:
            return nodes[:maxNodes].tolist()

        covMat = np.zeros((len(deadPixels), len(nodes)), dtype=bool)
        for j, n in enumerate(nodes):
            m = self._circle_mask(H, W, int(n[0]), int(n[1]), self.NodeCoverageRadius)
            covMat[:, j] = m[deadPixels[:, 0], deadPixels[:, 1]]

        selected = []
        uncovered = np.ones(len(deadPixels), dtype=bool)

        for _ in range(maxNodes):
            if not np.any(uncovered):
                break

            scores = np.sum(covMat & uncovered[:, None], axis=0)
            if np.max(scores) == 0:
                break

            best = np.argmax(scores)
            selected.append(best)
            uncovered &= ~covMat[:, best]
            covMat[:, best] = False

        return nodes[selected].tolist()

    def _computeCoverage(self, H, W, nodes, tx, ty, exclR):
        mask = self._circle_mask(H, W, tx, ty, exclR)
        for n in nodes:
            m = self._circle_mask(H, W, int(n[0]), int(n[1]), self.NodeCoverageRadius)
            mask |= m
        return mask

    def runBudget(self, result, maxBudgetNodes):
        nodes = result['finalNodes']
        H, W = result['deadZoneMask'].shape
        deadZoneMask = result['deadZoneMask']

        if len(nodes) <= maxBudgetNodes:
            return {
                'finalNodes': nodes,
                'coverageFraction': result['coverageFraction']
            }

        selected = self._optimizeILP(nodes.tolist(), H, W, deadZoneMask, maxBudgetNodes)

        covMask = self._computeCoverage(H, W, selected,
                                        result['transmitter'][0],
                                        result['transmitter'][1],
                                        result['bestR'])
        deadPixels = np.sum(deadZoneMask)
        covered = np.sum(deadZoneMask & covMask)
        coverageFraction = covered / deadPixels if deadPixels > 0 else 1.0

        return {
            'finalNodes': np.array(selected),
            'coverageFraction': coverageFraction
        }

print("✅ NodePlacerOptimizer ready!")

In [ ]:
# ╔══════════════════════════════════════════════════════════════════════════╗
# ║   CELL 5 — USER INPUTS  (input() prompts, validation, defaults)         ║
# ╚══════════════════════════════════════════════════════════════════════════╝


def _prompt_float(prompt, default, validator=None, error_msg=None):
    """Prompt for a float; blank input uses default. Repeats on invalid input."""
    while True:
        raw = input(f"{prompt} [{default}]: ").strip()
        try:
            val = default if raw == "" else float(raw)
        except ValueError:
            print(f"  ❌ Invalid number. Try again.")
            continue
        if validator and not validator(val):
            print(f"  ❌ {error_msg or 'Value out of range.'} Try again.")
            continue
        return val


def _prompt_int(prompt, default, validator=None, error_msg=None):
    while True:
        raw = input(f"{prompt} [{default}]: ").strip()
        try:
            val = default if raw == "" else int(raw)
        except ValueError:
            print(f"  ❌ Invalid integer. Try again.")
            continue
        if validator and not validator(val):
            print(f"  ❌ {error_msg or 'Value out of range.'} Try again.")
            continue
        return val


class _AmbiguousFrequency(Exception):
    """Raised when a bare (unsuffixed) number is given and could plausibly
    mean either GHz or MHz -- forces an explicit re-prompt instead of
    silently guessing, since guessing wrong here is a 1000x physics error
    that silently produces a meaningless simulation (this happened before:
    a bare "2.4" was silently parsed as 2.4 MHz instead of 2.4 GHz)."""
    def __init__(self, as_ghz, as_mhz):
        self.as_ghz = as_ghz
        self.as_mhz = as_mhz


def _parse_frequency(raw, default_hz=50e6):
    """'2.4GHz' -> 2.4e9; '915MHz' -> 915e6; '50MHz' -> 50e6.
    A BARE number with no suffix (e.g. "2.4" or "50") is never silently
    guessed -- it raises _AmbiguousFrequency so the caller can ask the user
    to confirm which unit they meant, since a wrong guess here is a 1000x
    physics error (a bare "50" could reasonably mean 50 MHz or 50 GHz;
    "2.4" could mean 2.4 GHz or, less plausibly, 2.4 MHz)."""
    raw = raw.strip()
    if raw == "":
        return default_hz
    s = raw.upper().replace(" ", "")
    if s.endswith("GHZ"):
        return float(s[:-3]) * 1e9
    if s.endswith("MHZ"):
        return float(s[:-3]) * 1e6
    if s.endswith("KHZ"):
        return float(s[:-3]) * 1e3
    if s.endswith("HZ"):
        return float(s[:-2])

    val = float(s)   # bare number, no suffix -- ambiguous, don't guess
    raise _AmbiguousFrequency(as_ghz=val * 1e9, as_mhz=val * 1e6)


def collect_user_inputs():
    print("=" * 72)
    print("  5G PHASED ARRAY COVERAGE SIMULATOR — REAL-TIME BEAM STEERING")
    print("=" * 72)

    cfg = {}

    print("\n─── Map Dimensions ───")
    cfg["width"] = _prompt_float("Map width (m)", 100.0, lambda v: v > 0,
                                  "Width must be > 0")
    cfg["height"] = _prompt_float("Map height (m)", 100.0, lambda v: v > 0,
                                   "Height must be > 0")

    print("\n─── Frequency ───")
    while True:
        raw = input("Frequency -- please include units, e.g. 50MHz, 2.4GHz, 915MHz [50MHz]: ").strip()
        try:
            cfg["freq_hz"] = _parse_frequency(raw, 50e6)
            if cfg["freq_hz"] <= 0:
                print("  ❌ Frequency must be positive.")
                continue
            print(f"  ✓ Parsed as {cfg['freq_hz']/1e9:.4f} GHz "
                  f"({cfg['freq_hz']/1e6:.1f} MHz).")
            break
        except _AmbiguousFrequency as amb:
            print(f"  ⚠️  '{raw}' has no unit. Did you mean "
                  f"{amb.as_ghz/1e9:.3g} GHz or {amb.as_mhz/1e6:.3g} MHz?")
            unit = input("  Type 'ghz' or 'mhz' to confirm: ").strip().lower()
            if unit in ("g", "ghz"):
                cfg["freq_hz"] = amb.as_ghz
                print(f"  ✓ Using {cfg['freq_hz']/1e9:.4f} GHz.")
                break
            elif unit in ("m", "mhz"):
                cfg["freq_hz"] = amb.as_mhz
                print(f"  ✓ Using {cfg['freq_hz']/1e6:.4f} MHz.")
                break
            else:
                print("  ❌ Not recognized -- please re-enter the frequency with units.")
        except ValueError:
            print("  ❌ Could not parse frequency. Examples: 50MHz, 2.4GHz, 915MHz")

    print("\n─── Transmit Power ───")
    cfg["single_antenna_tx_power_dbm"] = _prompt_float(
        "Single-antenna TX Power (dBm)", 20.0, lambda v: -30 <= v <= 36,
        "TX power must be between -30 and 36 dBm")
    cfg["array_element_tx_power_dbm"] = _prompt_float(
        "Phased-array PER-ELEMENT TX Power (dBm) -- this is what each of "
        "the 5 elements individually radiates, NOT the array's combined total",
        20.0, lambda v: -30 <= v <= 36,
        "TX power must be between -30 and 36 dBm")

    print("\n─── Antenna Position ───")
    cfg["antenna_x"] = _prompt_float(
        "Antenna X (m)", cfg["width"] / 2,
        lambda v: 0 <= v <= cfg["width"], f"X must be between 0 and {cfg['width']}")
    cfg["antenna_y"] = _prompt_float(
        "Antenna Y (m)", cfg["height"] / 2,
        lambda v: 0 <= v <= cfg["height"], f"Y must be between 0 and {cfg['height']}")

    print("\n─── Building Material ───")
    print("  1=Concrete  2=Glass  3=Brick  4=Reinforced  5=Custom")
    _MATERIALS = {
        "1": (3.5, 0.062, "Concrete"),
        "2": (6.0, 0.030, "Glass"),
        "3": (4.0, 0.080, "Brick"),
        "4": (8.0, 0.120, "Reinforced concrete"),
    }
    mat_choice = _prompt_int("Material [1-5]", 1, lambda v: 1 <= v <= 5,
                              "Choose 1-5")
    if str(mat_choice) in _MATERIALS:
        eps_r, sigma, mat_name = _MATERIALS[str(mat_choice)]
        print(f"  ✓ {mat_name}: εr={eps_r}, σ={sigma} S/m")
    else:
        eps_r = _prompt_float("  Custom εr", 3.5, lambda v: v >= 1,
                               "εr must be >= 1")
        sigma = _prompt_float("  Custom σ (S/m)", 0.01, lambda v: v > 0,
                               "σ must be > 0")
        mat_name = "Custom"
    cfg["eps_r"] = eps_r
    cfg["sigma"] = sigma
    cfg["material_name"] = mat_name
    # Map (eps_r, sigma) onto this notebook's alpha_air/alpha_eff absorption
    # model (same parametrization PhasedArraySim and SingleAntennaSim use).
    cfg["alpha_air"] = 0.00002
    cfg["alpha_eff"] = min(0.5, sigma * 6.0 + 0.02 * max(0.0, eps_r - 1))

    print("\n─── Grid Resolution ───")
    print("  Note: the simulation grid resolution is now determined by your")
    print("  uploaded mask/floorplan image's own pixel size (same convention")
    print("  as the phased-array solver) -- NOT by this PPW value. PPW here")
    print("  is only used as a target to check your uploaded mask against,")
    print("  and to warn you if it is too coarse for the chosen frequency.")
    cfg["ppw_target"] = _prompt_int("Target PPW (points/wavelength)", 10,
                                     lambda v: v >= 5, "PPW must be >= 5")

    print("\n─── PML / Boundary ───")
    cfg["pml_thickness"] = _prompt_float("PML thickness (m)", 0.5, lambda v: v > 0,
                                          "PML thickness must be > 0")

    print("\n─── Node Placement & Optimization (NodePlacerOptimizer) ───")
    print("  Dead zones are defined as the weakest X percentile of the map's")
    print("  own RSSI values (not a fixed dBm cutoff) -- this reliably finds")
    print("  several weak-spot clusters instead of just one or zero, since")
    print("  the threshold always adapts to this specific map's signal range.")
    cfg["dead_zone_percentile"] = _prompt_int(
        "Dead Zone Threshold (percentile)", 35, lambda v: 0 < v < 100,
        "Percentile must be between 0 and 100")
    cfg["node_coverage_radius"] = _prompt_int(
        "Node Coverage Radius (px)", 50, lambda v: v > 0,
        "Radius must be > 0")
    cfg["min_dist_pruning"] = _prompt_int(
        "Min Distance Pruning (px)", 70, lambda v: v > 0,
        "Must be > 0")
    cfg["min_cluster_area"] = _prompt_int(
        "Min Cluster Area (px)", 10, lambda v: v > 0,
        "Must be > 0")
    cfg["max_candidates"] = _prompt_int(
        "Max Candidates", 300, lambda v: v > 0,
        "Must be > 0")
    cfg["max_nodes"] = _prompt_int(
        "Max Nodes (0 = unlimited)", 0, lambda v: v >= 0,
        "Must be >= 0")
    cfg["dbm_min"] = _prompt_float("Minimum dBm (colorbar/display scale)", -80.0)
    cfg["dbm_max"] = _prompt_float("Maximum dBm (colorbar/display scale)", 10.0)

    print("\n─── Target Eligibility ───")
    print("  NodePlacerOptimizer places nodes wherever it finds dead zones,")
    print("  but a placed node's own signal might still be decent (it's a")
    print("  good COVERAGE point, not necessarily the single worst pixel).")
    print("  This threshold decides which placed nodes are actually weak")
    print("  enough to be worth steering the beam toward.")
    cfg["target_rssi_threshold"] = _prompt_float(
        "Target RSSI threshold (dBm) -- only nodes below this get targeted",
        -80.0)

    print("\n─── Real-Time Loop ───")
    cfg["n_iterations"] = _prompt_int("Iterations N", 10, lambda v: v >= 1,
                                       "N must be >= 1")

    print("\n─── Output ───")
    out_dir = input("Output dir [./frames]: ").strip() or "./frames"
    os.makedirs(out_dir, exist_ok=True)
    cfg["output_dir"] = out_dir
    cfg["video_fps"] = _prompt_int(
        "Video FPS (frames per second for the output video)", 10,
        lambda v: v >= 1, "FPS must be >= 1")

    print("\n" + "=" * 72)
    print("  PARAMETER SUMMARY")
    print("=" * 72)
    for k, v in cfg.items():
        print(f"  {k:18s}: {v}")
    print("=" * 72)

    return cfg


def upload_map_and_mask():
    """Upload floorplan.png and mask.png. mask: black=buildings, white=air."""
    if not IN_COLAB:
        raise RuntimeError(
            "File upload requires Google Colab (google.colab.files). "
            "If running locally, load `img` (PIL.Image) and `mask` "
            "(np.ndarray, 1=building/0=air) yourself before continuing.")

    print("\nUpload the FLOORPLAN image (floorplan.png):")
    up_map = files.upload()
    if not up_map:
        raise ValueError("No floorplan uploaded.")
    map_name = list(up_map.keys())[0]
    img = Image.open(io.BytesIO(up_map[map_name])).convert("RGB")

    print("\nUpload the MASK image (mask.png) — black=buildings, white=air:")
    up_mask = files.upload()
    if not up_mask:
        raise ValueError("No mask uploaded.")
    mask_name = list(up_mask.keys())[0]
    mask_img = Image.open(io.BytesIO(up_mask[mask_name])).convert("L")
    mask_array = np.array(mask_img)
    mask = (mask_array < 128).astype(int)   # black(<128) -> 1 (building)

    print(f"✓ Floorplan: {img.size[0]}×{img.size[1]} px   "
          f"Mask: {mask.shape[1]}×{mask.shape[0]} px")
    return img, mask


print("✅  Input-collection helpers loaded. Call collect_user_inputs() and "
      "upload_map_and_mask() in the next cell.")


In [ ]:
# ╔══════════════════════════════════════════════════════════════════════════╗
# ║   CELL 6 — NODE PLACEMENT + FRAME GENERATION                            ║
# ║                                                                          ║
# ║   Uses the NEW NodePlacerOptimizer (per-cluster placement + prune/ILP   ║
# ║   optimization, percentile-based dead-zone threshold). With             ║
# ║   ImageScaleFactor=1.0 and mask.shape == rssi_dbm_grid.shape (always    ║
# ║   true here), result['finalNodes'] / result['transmitter'] come back    ║
# ║   directly in this notebook's one shared (rows, cols) grid — no         ║
# ║   separate img-space conversion needed anywhere.                        ║
# ╚══════════════════════════════════════════════════════════════════════════╝


def run_node_placement(img, mask, rssi_dbm_grid, cfg):
    """
    Run NodePlacerOptimizer (new class) on the single-antenna baseline
    coverage. With ImageScaleFactor=1.0 and mask.shape == rssi_dbm_grid.shape
    (always true in this notebook -- one grid everywhere), the new class's
    internal cv2.resize calls are no-ops: result['img'] and every node
    position come back in EXACTLY the same (rows, cols) grid as
    rssi_dbm_grid and mask. No img-space<->solver-space conversion needed
    anywhere -- one coordinate system, confirmed by inspection of the class.

    img              : PIL.Image (uploaded floorplan)
    mask             : np.ndarray, SAME (rows, cols) as rssi_dbm_grid
                        (1=building, 0=air)
    rssi_dbm_grid    : np.ndarray from SingleAntennaSim.rssi_dbm_grid()
    cfg              : dict with the NodePlacerOptimizer parameters
                        collected in collect_user_inputs().
    """
    solver_rows, solver_cols = rssi_dbm_grid.shape

    # The FDM solvers (Cell 2/Cell 3) use a 20px "sponge" absorbing boundary
    # layer around all four edges of the grid, to stop outgoing waves from
    # reflecting back off the map's edges. That border is DELIBERATELY made
    # very lossy -- it's a numerical technique, not real-world signal -- so
    # it always reads as extremely weak dBm. Left as-is, that fake weak
    # ring gets treated as the "worst" part of the map: it skews the
    # percentile threshold to be far more pessimistic than the interior
    # data actually is, AND NodePlacerOptimizer happily places nodes there
    # (they're in "air"), which then never stop winning "weakest node"
    # searches since nothing beats a numerical artifact. Both issues are
    # fixed by excluding this border from consideration entirely:
    SPONGE_THICKNESS = 20   # must match cur_cell2_phasedarray.py / cur_cell3_singleantenna.py
    interior = np.zeros_like(mask, dtype=bool)
    interior[SPONGE_THICKNESS:-SPONGE_THICKNESS, SPONGE_THICKNESS:-SPONGE_THICKNESS] = True

    p = cfg["_single_sim_params"]
    src_row, src_col = meters_to_pixel(p["antenna_x_m"], p["antenna_y_m"],
                                        p["h"], p["hy"])
    src_row = int(np.clip(src_row, 0, solver_rows - 1))
    src_col = int(np.clip(src_col, 0, solver_cols - 1))

    # Mark the border as "building" (not air) so NodePlacerOptimizer's own
    # existing air-only placement rule naturally excludes it -- no new
    # exclusion mechanism needed, just feed it accurate information.
    mask_for_placement = mask.copy()
    mask_for_placement[~interior] = 1

    npo = NodePlacerOptimizer()
    npo.ImageScaleFactor = 1.0   # keep everything on one shared grid
    npo.NodeCoverageRadius = cfg["node_coverage_radius"]
    # Percentile computed over the INTERIOR only, so the sponge border's
    # fake extreme values can't skew what "the weakest 35%" even means.
    npo.DeadZoneThreshold_dBm = np.percentile(rssi_dbm_grid[interior].flatten(),
                                               cfg["dead_zone_percentile"])
    npo.MinDistPruning = cfg["min_dist_pruning"]
    npo.MinClusterAreaPx = cfg["min_cluster_area"]
    npo.RadiiToTest = np.arange(10, 81, 10)
    npo.MaxCandidates = cfg["max_candidates"]
    npo.MaxNodes = cfg["max_nodes"]

    print(f"   Dead-zone threshold: {npo.DeadZoneThreshold_dBm:.1f} dBm "
          f"({cfg['dead_zone_percentile']}th percentile of this map's interior "
          f"RSSI, excluding the {SPONGE_THICKNESS}px absorbing boundary)")

    img_array = np.array(img)
    result = npo.run(
        campusImg=img_array,
        stage1_out={"mag_db": rssi_dbm_grid, "src_x": src_col, "src_y": src_row},
        binaryMask=mask_for_placement
    )


    all_nodes = []
    for idx, (nx, ny) in enumerate(result["finalNodes"]):
        col = int(np.clip(round(nx), 0, solver_cols - 1))
        row = int(np.clip(round(ny), 0, solver_rows - 1))
        all_nodes.append({
            "id": idx,
            "_solver_rc": (row, col),
            "baseline_rssi": float(rssi_dbm_grid[row, col]),
        })

    # NodePlacerOptimizer decides WHERE to place nodes (percentile-based
    # dead-zone detection, then picks good coverage-representative points --
    # not necessarily the single worst pixel in a cluster). Separately, the
    # user decides which of those PLACED nodes are actually weak enough to
    # be worth steering the beam toward, via an explicit dBm cutoff.
    weak_nodes = [n for n in all_nodes if n["baseline_rssi"] < cfg["target_rssi_threshold"]]

    print(f"✓ NodePlacer: {len(result['nodesBeforeOpt'])} node(s) before "
          f"optimization -> {len(all_nodes)} after pruning "
          f"({result['coverageFraction']*100:.1f}% dead-zone coverage), "
          f"{len(weak_nodes)} below your {cfg['target_rssi_threshold']} dBm target threshold.")

    return {
        "result": result,
        "all_nodes": all_nodes,
        "weak_nodes": weak_nodes,
        "src_row": src_row,
        "src_col": src_col,
    }


# ═══════════════════════════════════════════════════════════════════════════
#  FRAME GENERATION — all drawn directly in the solver's (rows, cols) grid,
#  displayed with extent=[0, real_width, real_height, 0] so axis labels show
#  meters. Node/antenna positions converted once via
#  node_img_px_to_solver_px / meters_to_pixel — no other conversions.
# ═══════════════════════════════════════════════════════════════════════════
FRAME_FIGSIZE = (12, 10)
FRAME_DPI = 100
FRAME_FACECOLOR = "#1e1e2e"


def _draw_common_frame(mag_db, W, H, vmin=-100, vmax=-30):
    fig, ax = plt.subplots(figsize=FRAME_FIGSIZE, dpi=FRAME_DPI,
                            facecolor=FRAME_FACECOLOR)
    ax.set_facecolor(FRAME_FACECOLOR)
    im = ax.imshow(mag_db, cmap="jet", extent=[0, W, H, 0], vmin=vmin, vmax=vmax)
    cb = fig.colorbar(im, ax=ax, fraction=0.04, pad=0.03)
    cb.set_label("RSSI (dBm)", color="#cdd6f4")
    plt.setp(cb.ax.yaxis.get_ticklabels(), color="#cdd6f4")
    ax.tick_params(colors="#cdd6f4")
    for sp in ax.spines.values():
        sp.set_edgecolor("#45475a")
    return fig, ax


def _rc_to_m(row, col, h, hy):
    return col * h, row * hy


def _text_box(ax, text):
    ax.text(0.02, 0.98, text, transform=ax.transAxes, va="top", ha="left",
            fontsize=10, family="monospace", color="#cdd6f4",
            bbox=dict(facecolor="#1e1e2e", alpha=0.85, pad=6,
                       boxstyle="round,pad=0.4"))


def generate_node_placement_frame(mask, all_nodes, weak_nodes,
                                   src_row, src_col, W, H, h, hy, out_path):
    """
    Frame 0 — shown BEFORE the baseline RSSI heatmap. Displays just the
    building layout (mask) with every placed node marked (red = above
    threshold, orange = below threshold / weak) and the antenna position,
    so the node-placement result is visible on its own before any coverage
    heatmap is drawn.
    """
    fig, ax = plt.subplots(figsize=FRAME_FIGSIZE, dpi=FRAME_DPI,
                            facecolor=FRAME_FACECOLOR)
    ax.set_facecolor(FRAME_FACECOLOR)

    # Building mask as a simple grayscale backdrop (1=building, 0=air).
    backdrop = np.where(mask == 1, 0.25, 0.05)
    ax.imshow(backdrop, cmap="gray", extent=[0, W, H, 0], vmin=0, vmax=1)

    weak_ids = {n["id"] for n in weak_nodes}
    for n in all_nodes:
        nr, nc = n["_solver_rc"]
        nx_m, ny_m = _rc_to_m(nr, nc, h, hy)
        color = "orange" if n["id"] in weak_ids else "red"
        ax.scatter(nx_m, ny_m, s=110, c=color, edgecolors="white",
                   linewidths=1.2, zorder=5)
        ax.text(nx_m + 0.015 * W, ny_m, f"#{n['id']:02d}", color="white",
                fontsize=8, zorder=6, va="center")

    ax_m, ay_m = _rc_to_m(src_row, src_col, h, hy)
    ax.scatter(ax_m, ay_m, s=400, marker="*", c="magenta",
               edgecolors="white", linewidths=1.2, zorder=8)

    ax.set_xlim(0, W)
    ax.set_ylim(H, 0)
    ax.tick_params(colors="#cdd6f4")
    for sp in ax.spines.values():
        sp.set_edgecolor("#45475a")

    ax.scatter([], [], s=110, c="red", edgecolors="white", label="Node (OK)")
    ax.scatter([], [], s=110, c="orange", edgecolors="white",
               label=f"Node (weak, will need coverage)")
    ax.scatter([], [], s=400, marker="*", c="magenta", edgecolors="white",
               label="Antenna")
    leg = ax.legend(loc="lower right", facecolor="#1e1e2e", labelcolor="#cdd6f4",
                     framealpha=0.85, fontsize=9)

    _text_box(ax, f"NODE PLACEMENT — {len(all_nodes)} node(s) placed\n"
                   f"{len(weak_nodes)} below RSSI threshold (need coverage)\n"
                   f"→ Baseline coverage heatmap next")

    plt.tight_layout()
    fig.savefig(out_path, facecolor=FRAME_FACECOLOR)
    plt.close(fig)


def generate_baseline_frame(rssi_dbm_grid, all_nodes, weakest_node,
                             src_row, src_col, W, H, h, hy, out_path,
                             dbm_min=-100, dbm_max=-30):
    fig, ax = _draw_common_frame(rssi_dbm_grid, W, H, vmin=dbm_min, vmax=dbm_max)

    for n in all_nodes:
        nr, nc = n["_solver_rc"]
        nx_m, ny_m = _rc_to_m(nr, nc, h, hy)
        ax.scatter(nx_m, ny_m, s=80, c="red", edgecolors="white",
                   linewidths=1.0, zorder=5)

    ax_m, ay_m = _rc_to_m(src_row, src_col, h, hy)

    if weakest_node is not None:
        wr, wc = weakest_node["_solver_rc"]
        wx_m, wy_m = _rc_to_m(wr, wc, h, hy)
        ax.scatter(wx_m, wy_m, s=400, c="yellow", alpha=0.3, zorder=4)
        ax.scatter(wx_m, wy_m, s=200, c="yellow", edgecolors="white",
                   linewidths=1.5, zorder=6)
        ax.annotate("", xy=(wx_m, wy_m), xytext=(ax_m, ay_m),
                    arrowprops=dict(arrowstyle="->", color="cyan", lw=2), zorder=7)
        _text_box(ax, f"BASELINE — Single Antenna\n"
                       f"Target: #{weakest_node['id']:02d} | "
                       f"RSSI: {weakest_node['baseline_rssi']:.1f} dBm\n"
                       f"→ Replacing with phased array")
    else:
        _text_box(ax, "BASELINE — Single Antenna\nNo weak spots found.")

    ax.scatter(ax_m, ay_m, s=400, marker="*", c="magenta",
               edgecolors="white", linewidths=1.2, zorder=8)

    plt.tight_layout()
    fig.savefig(out_path, facecolor=FRAME_FACECOLOR)
    plt.close(fig)


def generate_final_labeled_frame(mag_db, theta_current, all_nodes,
                                  unreachable_ids, antenna_m, W, H, out_path):
    """
    Repeats the very last steering frame (same beam angle/heatmap, same
    beam wedge and antenna), but labels each node with the dB value read
    directly from THIS heatmap at that node's position -- what the phased
    array is actually delivering to each node at this final beam angle,
    not the static single-antenna baseline. Grey nodes (permanently
    excluded/unreachable) are shown grey, same as during steering. Shown
    once, right before the max-hold frame.
    """
    fig, ax = _draw_common_frame(mag_db, W, H, vmin=np.min(mag_db),
                                  vmax=max(15, np.max(mag_db)))

    for n in all_nodes:
        nx_m, ny_m = n["_m"]
        nr, nc = n["_solver_rc"]
        val_at_this_angle = mag_db[nr, nc]
        color = "grey" if n["id"] in unreachable_ids else "red"
        ax.scatter(nx_m, ny_m, s=80, c=color, edgecolors="white",
                   linewidths=1.0, zorder=5)
        ax.annotate(f"{val_at_this_angle:.1f}", (nx_m, ny_m),
                    textcoords="offset points", xytext=(0, 8),
                    ha="center", fontsize=7, color="white", zorder=9,
                    path_effects=[pe.withStroke(linewidth=2, foreground="black")])

    wedge = Wedge(antenna_m, 0.15 * min(W, H), theta_current - 25,
                  theta_current + 25, facecolor="#00ffff", alpha=0.12, zorder=3)
    ax.add_patch(wedge)

    ax.scatter(*antenna_m, s=400, marker="*", c="magenta",
               edgecolors="white", linewidths=1.2, zorder=8)

    beam_len = 0.1 * min(W, H)
    dx = beam_len * math.cos(math.radians(theta_current))
    dy = -beam_len * math.sin(math.radians(theta_current))
    ax.annotate("", xy=(antenna_m[0]+dx, antenna_m[1]+dy), xytext=antenna_m,
                arrowprops=dict(arrowstyle="->", color="white", lw=2), zorder=8)

    _text_box(ax, f"FINAL BEAM POSITION — {theta_current:03.0f}°\n"
                   f"Per-node dB labeled at this angle\n"
                   f"{len(all_nodes)} node(s) shown, {len(unreachable_ids)} grey/unreachable")

    plt.tight_layout()
    fig.savefig(out_path, facecolor=FRAME_FACECOLOR)
    plt.close(fig)


def generate_max_hold_frame(max_hold_db, all_nodes, unreachable_ids,
                             antenna_m, W, H, out_path):
    """
    Final frame: full 360° max-hold coverage map -- the best (highest) dB
    value at each pixel across all 73 precomputed beam angles (matches the
    original PhasedArraySim.max_hold_coverage() exactly). Shown once, as the
    very last frame, after the steering loop completes. Grey = permanently
    excluded/unreachable node; red = everyone else still searchable.
    """
    fig, ax = _draw_common_frame(max_hold_db, W, H,
                                  vmin=np.min(max_hold_db),
                                  vmax=max(15, np.max(max_hold_db)))

    for n in all_nodes:
        nx_m, ny_m = n["_m"]
        color = "grey" if n["id"] in unreachable_ids else "red"
        ax.scatter(nx_m, ny_m, s=80, c=color, edgecolors="white",
                   linewidths=1.0, zorder=5)

    ax.scatter(*antenna_m, s=400, marker="*", c="magenta",
               edgecolors="white", linewidths=1.2, zorder=8)

    _text_box(ax, f"MAX-HOLD COVERAGE — full 360° sweep\n"
                   f"Best signal per pixel across all beam angles\n"
                   f"{len(all_nodes)} node(s) total, {len(unreachable_ids)} grey/unreachable")

    plt.tight_layout()
    fig.savefig(out_path, facecolor=FRAME_FACECOLOR)
    plt.close(fig)


def generate_frame(mag_db, theta_current, theta_target, target, all_nodes,
                    unreachable_ids, iteration, n_iterations, antenna_m, W, H,
                    out_path, is_hold=False):
    """
    One steering-loop frame. Colors:
      - grey   : node was targeted directly before and still stayed below
                 the user's RSSI threshold -- permanently excluded, out of
                 our coverage, never searched again.
      - yellow : the current target.
      - red    : everyone else (still searchable, no permanent "served"
                 state -- can be retargeted any time it's the weakest).
    """
    fig, ax = _draw_common_frame(mag_db, W, H, vmin=np.min(mag_db),
                                  vmax=max(15, np.max(mag_db)))

    for n in all_nodes:
        nx_m, ny_m = n["_m"]
        if n["id"] in unreachable_ids:
            ax.scatter(nx_m, ny_m, s=80, c="grey", edgecolors="white",
                       linewidths=1.0, zorder=5)
        elif n["id"] == target["id"]:
            ax.scatter(nx_m, ny_m, s=200, c="yellow", edgecolors="white",
                       linewidths=1.5, zorder=6)
        else:
            ax.scatter(nx_m, ny_m, s=80, c="red", edgecolors="white",
                       linewidths=1.0, zorder=5)

    wedge = Wedge(antenna_m, 0.15 * min(W, H), theta_current - 25,
                  theta_current + 25, facecolor="#00ffff", alpha=0.12, zorder=3)
    ax.add_patch(wedge)

    ax.scatter(*antenna_m, s=400, marker="*", c="magenta",
               edgecolors="white", linewidths=1.2, zorder=8)

    beam_len = 0.1 * min(W, H)
    dx = beam_len * math.cos(math.radians(theta_current))
    dy = -beam_len * math.sin(math.radians(theta_current))
    ax.annotate("", xy=(antenna_m[0]+dx, antenna_m[1]+dy), xytext=antenna_m,
                arrowprops=dict(arrowstyle="->", color="white", lw=2), zorder=8)

    hold_tag = " [HOLD]" if is_hold else ""
    _text_box(ax, f"Iteration: {iteration:02d} / {n_iterations}{hold_tag}\n"
                   f"Beam: {theta_current:03.0f}° → {theta_target:03.0f}°\n"
                   f"Target: #{target['id']:02d} | Baseline: {target['baseline_rssi']:.1f} dBm "
                   f"| Was: {target['current_rssi']:.1f} dBm")

    plt.tight_layout()
    fig.savefig(out_path, facecolor=FRAME_FACECOLOR)
    plt.close(fig)


def compile_video_from_frames(output_dir, fps, video_path=None):
    """
    Stitch every frame_XXXX.png in output_dir (in order) into an MP4 video
    at the given fps. The PNGs are kept as-is -- this just adds a video on
    top, it doesn't replace or delete the individual frame images.
    """
    if video_path is None:
        video_path = os.path.join(output_dir, "simulation_video.mp4")

    frame_files = sorted(
        f for f in os.listdir(output_dir)
        if f.startswith("frame_") and f.endswith(".png")
    )
    if not frame_files:
        print("   ⚠️  No frames found -- skipping video compilation.")
        return None

    first_frame = cv2.imread(os.path.join(output_dir, frame_files[0]))
    height, width = first_frame.shape[:2]

    fourcc = cv2.VideoWriter_fourcc(*"mp4v")
    writer = cv2.VideoWriter(video_path, fourcc, fps, (width, height))

    for fname in frame_files:
        frame = cv2.imread(os.path.join(output_dir, fname))
        if frame.shape[:2] != (height, width):
            frame = cv2.resize(frame, (width, height))
        writer.write(frame)
    writer.release()

    print(f"   ✓ Video written: {video_path} "
          f"({len(frame_files)} frames @ {fps} fps, "
          f"{len(frame_files)/fps:.1f}s runtime)")
    return video_path


print("✅  Node placement + frame generation functions loaded.")


In [ ]:
# ╔══════════════════════════════════════════════════════════════════════════╗
# ║   CELL 7 — MAIN LOOP                                                    ║
# ║                                                                          ║
# ║   Pipeline:                                                             ║
# ║     1. Single-antenna solve -> baseline RSSI (true dBm)                 ║
# ║     2. NodePlacerOptimizer -> weak-spot nodes (percentile-based)        ║
# ║     3. Single antenna REMOVED; PhasedArraySim installed at SAME         ║
# ║        position; all 73 beam angles precomputed once                   ║
# ║     4. LOOP, every iteration:                                          ║
# ║          a. search all still-searchable weak nodes (excludes grey/     ║
# ║             unreachable ones) fresh for the current weakest one --     ║
# ║             no permanent "served" state otherwise: a node visited      ║
# ║             before can win again, since the beam only covers one       ║
# ║             place at a time and a previous target isn't fixed          ║
# ║          b. INSTANT electronic steering -- a phased array changes its  ║
# ║             phase shifters, it doesn't move mechanically, so the beam  ║
# ║             jumps straight to the target angle. One frame is written,  ║
# ║             already showing the beam pointed at the target -- no 5°    ║
# ║             step animation.                                            ║
# ║          c. check the RSSI actually delivered at the target from THIS  ║
# ║             beam position. If still below the user's target threshold  ║
# ║             even when pointed directly at it, the array can't fix it   ║
# ║             -- mark it GREY and permanently exclude it from now on     ║
# ║             (the one exception to "never permanently exclude")         ║
# ║          d. go back to (a) -- search again from scratch                ║
# ║        runs for n_iterations (or stops early if every node goes grey), ║
# ║        then writes a final labeled frame + a max-hold frame            ║
# ╚══════════════════════════════════════════════════════════════════════════╝


def main():
    cfg = collect_user_inputs()
    img, mask = upload_map_and_mask()

    W, H = cfg["width"], cfg["height"]
    rows, cols = mask.shape
    h  = W / cols     # meters/pixel, x — SAME for every solver in this notebook
    hy = H / rows     # meters/pixel, y

    # ── PPW diagnostic against the ACTUAL uploaded mask resolution ─────────
    lam = 3e8 / cfg["freq_hz"]
    ppw_actual = lam / min(h, hy)
    print(f"\nMask resolution: {cols}×{rows} px over {W}×{H} m "
          f"-> grid spacing {min(h,hy):.3f} m -> {ppw_actual:.1f} points/wavelength "
          f"at {cfg['freq_hz']/1e6:.1f} MHz.")
    if ppw_actual < cfg["ppw_target"]:
        print(f"  ⚠️  This is below your target PPW ({cfg['ppw_target']}). "
              f"Consider a higher-resolution mask image or a lower frequency "
              f"for a sharper simulation. Continuing with the mask's native "
              f"resolution (same grid the phased array will use).")

    # ── 1. Single-antenna baseline ──────────────────────────────────────
    print("\n" + "=" * 72)
    print("STEP 1: Single-antenna baseline solve")
    print("=" * 72)
    single_sim = SingleAntennaSim(
        binary_mask=mask, real_width=W, real_height=H,
        tx_power_dbm=cfg["single_antenna_tx_power_dbm"], tx_gain_dbi=2.15, rx_gain_dbi=0.0,
        antenna_x=cfg["antenna_x"], antenna_y=cfg["antenna_y"],
        f_sim=cfg["freq_hz"], alpha_air=cfg["alpha_air"], alpha_eff=cfg["alpha_eff"])
    single_sim.place_source()
    rssi_grid = single_sim.rssi_dbm_grid()
    cfg["_single_sim_params"] = single_sim.params   # antenna row/col for node placement

    # ── 2. Node placement on baseline coverage ──────────────────────────
    print("\n" + "=" * 72)
    print("STEP 2: Node placement (NodePlacerOptimizer)")
    print("=" * 72)
    placement = run_node_placement(img, mask, rssi_grid, cfg)
    all_nodes = placement["all_nodes"]
    weak_nodes = placement["weak_nodes"]
    src_row, src_col = placement["src_row"], placement["src_col"]

    # _solver_rc is already set by run_node_placement (new NodePlacerOptimizer
    # returns node positions directly in this notebook's one shared grid).
    # Just add physical meters for scatter placement on frames. Also track
    # each node's CURRENT known RSSI, starting as the static single-antenna
    # baseline -- this gets updated to the real delivered value every time
    # a node is actually targeted (see the steering loop), so "search for
    # the weakest" always reflects the freshest known info, not a frozen
    # number that never improves even after the array successfully fixes
    # that node's coverage.
    for n in all_nodes:
        rc = n["_solver_rc"]
        n["_m"] = _rc_to_m(rc[0], rc[1], h, hy)
        n["current_rssi"] = n["baseline_rssi"]

    # ── 2b. FIRST frame: node placements only (before any coverage heatmap
    # or beam steering) — shows where NodePlacerOptimizer put nodes. ───────
    print("\n" + "=" * 72)
    print("STEP 2b: Node-placement frame (frame_0000)")
    print("=" * 72)
    frame_num = 0
    generate_node_placement_frame(
        mask, all_nodes, weak_nodes, src_row, src_col, W, H, h, hy,
        os.path.join(cfg["output_dir"], f"frame_{frame_num:04d}.png"))
    frame_num += 1

    if not weak_nodes:
        if not all_nodes:
            print(f"\n  ✅ No dead-zone nodes found at the "
                  f"{cfg['dead_zone_percentile']}th percentile threshold — "
                  f"coverage already looks adequate. Nothing to steer toward. "
                  f"(Try a higher percentile if you expected some.)")
        else:
            print(f"\n  ✅ NodePlacer placed {len(all_nodes)} node(s), but "
                  f"none are below your target threshold of "
                  f"{cfg['target_rssi_threshold']} dBm (weakest placed node: "
                  f"{min(n['baseline_rssi'] for n in all_nodes):.1f} dBm). "
                  f"Nothing to steer toward. (Try a less negative target "
                  f"threshold, e.g. -60, if you expected some.)")
        antenna_m = _rc_to_m(src_row, src_col, h, hy)
        generate_baseline_frame(
            rssi_grid, all_nodes, None, src_row, src_col, W, H, h, hy,
            os.path.join(cfg["output_dir"], f"frame_{frame_num:04d}.png"),
            dbm_min=cfg["dbm_min"], dbm_max=cfg["dbm_max"])
        frame_num += 1
        video_path = compile_video_from_frames(cfg["output_dir"], cfg["video_fps"])
        print(f"\n  ✅ DONE — {frame_num} frame(s) written to {cfg['output_dir']}")
        if video_path:
            print(f"     Video: {video_path}")
        return

    weakest = min(weak_nodes, key=lambda n: n["current_rssi"])

    print("\n" + "=" * 72)
    print("STEP 3: Baseline frame")
    print("=" * 72)
    generate_baseline_frame(
        rssi_grid, all_nodes, weakest, src_row, src_col, W, H, h, hy,
        os.path.join(cfg["output_dir"], f"frame_{frame_num:04d}.png"),
        dbm_min=cfg["dbm_min"], dbm_max=cfg["dbm_max"])
    frame_num += 1

    # ── 3. Phased array setup — SAME position, SAME grid (mask.shape) ──────
    print("\n" + "=" * 72)
    print("STEP 4: Phased array setup at same antenna position (single antenna removed)")
    print("=" * 72)
    del single_sim   # free single-antenna arrays before phased-array setup

    phased = PhasedArraySim(
        raw_image=img, binary_mask=mask, real_width=W, real_height=H,
        tx_power_dbm=cfg["array_element_tx_power_dbm"],
        antenna_x=cfg["antenna_x"], antenna_y=cfg["antenna_y"],
        f_sim=cfg["freq_hz"], alpha_air=cfg["alpha_air"],
        alpha_eff=cfg["alpha_eff"], step_angle=5)
    phased.process_map()
    phased.place_sources()
    phased._build_system_matrix()

    # Precompute ALL 73 angle frames (0°,5°,...,355°) ONCE, up front -- exactly
    # like the original PhasedArraySim's sweep_and_solve(). Every beam step
    # below just looks up an already-computed frame instead of re-solving.
    phased.precompute_all_angles()

    antenna_m = _rc_to_m(src_row, src_col, h, hy)

    # ── 4. Real-time steering loop (instant electronic steering) ───────────
    # A phased array steers by changing phase shifters -- there's no
    # mechanical movement, so the beam jumps directly to the target angle.
    # No 5°-step animation: one frame per iteration, showing the beam
    # already pointed at the target.
    print("\n" + "=" * 72)
    print("STEP 5: Real-time beam steering loop (instant electronic steering)")
    print("=" * 72)
    N = cfg["n_iterations"]
    theta_current = 0.0
    unreachable_ids = set()   # nodes that stayed below threshold even once
                               # targeted directly -- permanently excluded,
                               # the one exception to "never permanently
                               # exclude a node".

    for iteration in range(1, N + 1):
        searchable = [n for n in weak_nodes if n["id"] not in unreachable_ids]
        if not searchable:
            print(f"  All weak nodes are unreachable (grey) -- stopping early "
                  f"after {iteration - 1} iteration(s).")
            break

        # (a) search fresh for the weakest searchable node, using each
        # node's CURRENT known RSSI (updated below every time a node is
        # actually targeted) -- not the frozen single-antenna baseline.
        # Otherwise a node that's already been fixed by the array (now
        # delivering a healthy signal) would keep "winning" forever just
        # because its original baseline number never changes, starving
        # other genuinely-still-weak nodes of a turn.
        target = min(searchable, key=lambda n: n["current_rssi"])
        tx_m, ty_m = target["_m"]

        # Bearing from antenna to target (display convention: y increases
        # downward, so flip for standard math atan2 where +y is up).
        dx = tx_m - antenna_m[0]
        dy = -(ty_m - antenna_m[1])
        theta_target_raw = math.degrees(math.atan2(dy, dx)) % 360
        theta_target, mag_db = phased.get_cached_angle(theta_target_raw)
        theta_current = theta_target

        # (b) the beam is ALREADY at the target -- one frame, no stepping.
        generate_frame(
            mag_db, theta_current, theta_target, target, all_nodes,
            unreachable_ids, iteration, N, antenna_m, W, H,
            os.path.join(cfg["output_dir"], f"frame_{frame_num:04d}.png"),
            is_hold=True)
        frame_num += 1

        # (c) check the RSSI actually delivered at this node, from THIS
        # beam position, and record it as this node's freshest known value
        # -- this is what future "weakest" searches will compare against.
        # If still below the user's threshold even when pointed directly
        # at it, the array can't fix it -- mark grey and permanently
        # exclude it (the one exception to "never exclude").
        nr, nc = target["_solver_rc"]
        delivered_rssi = float(mag_db[nr, nc])
        target["current_rssi"] = delivered_rssi
        if delivered_rssi < cfg["target_rssi_threshold"]:
            unreachable_ids.add(target["id"])
            print(f"  Iteration {iteration}/{N}: targeted node #{target['id']:02d} "
                  f"-> still {delivered_rssi:.1f} dBm (below "
                  f"{cfg['target_rssi_threshold']} dBm threshold) -- "
                  f"marked GREY, excluded from now on.")
        else:
            print(f"  Iteration {iteration}/{N}: targeted node #{target['id']:02d} "
                  f"-> {delivered_rssi:.1f} dBm, now above threshold.")
        # (d) loop back to (a) — search again from scratch next iteration

    # ── 5. Final labeled frame: last beam position, RSSI written per node ──
    print("\n" + "=" * 72)
    print("STEP 6: Final labeled frame (last beam position + per-node dB)")
    print("=" * 72)
    _, final_mag_db = phased.get_cached_angle(theta_current)
    generate_final_labeled_frame(
        final_mag_db, theta_current, all_nodes, unreachable_ids, antenna_m, W, H,
        os.path.join(cfg["output_dir"], f"frame_{frame_num:04d}.png"))
    frame_num += 1
    print(f"  Final labeled frame written (angle {theta_current:.0f}°, "
          f"dB labeled above each node).")

    # ── 6. Final frame: full 360° max-hold coverage map ─────────────────────
    print("\n" + "=" * 72)
    print("STEP 7: Max-hold coverage frame (final frame)")
    print("=" * 72)
    max_hold_db = phased.max_hold_coverage()
    generate_max_hold_frame(
        max_hold_db, all_nodes, unreachable_ids, antenna_m, W, H,
        os.path.join(cfg["output_dir"], f"frame_{frame_num:04d}.png"))
    frame_num += 1
    print(f"  Max-hold frame written (best signal per pixel across all 73 angles).")

    # ── 7. Compile all frames into a video ──────────────────────────────────
    print("\n" + "=" * 72)
    print("STEP 8: Compiling video from frames")
    print("=" * 72)
    video_path = compile_video_from_frames(cfg["output_dir"], cfg["video_fps"])

    print("\n" + "=" * 72)
    print(f"  ✅ DONE — {frame_num} frame images + video written to {cfg['output_dir']}")
    if video_path:
        print(f"     Video: {video_path}")
    print("=" * 72)


if __name__ == "__main__":
    main()
